# Windowed UID Experiment

Active/Passive alternations — windowed token-level UID analysis.

**Before running:** Set runtime to A100 via *Runtime → Change runtime type → A100*.

## Cell 1 — GPU check

In [ ]:
import torch
print('GPU: ', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
print('CUDA:', torch.version.cuda)
print('bf16:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

## Cell 2 — Install uv, clone repo, set working directory

`%cd` persists the directory for all subsequent cells. `!` runs a shell command.

In [ ]:
import os
!pip install uv -q
if os.path.exists('/content/repo'):
    # Already cloned — just pull latest changes
    !git -C /content/repo pull origin colab-monorepo
else:
    !git clone https://github.com/NolanChai/active-passive-alternations.git /content/repo
    !git -C /content/repo checkout colab-monorepo
%cd /content/repo

## Cell 3 — Install dependencies

`uv` will download Python 3.13 if needed (~1 min first time), then install packages.

In [ ]:
!uv sync

## Cell 4 — Mount data

Use **Option A** (Google Drive) or **Option B** (direct upload). Run only one.

In [ ]:
# Option A: Google Drive
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/YOUR_DATA_FOLDER'  # <-- adjust this path
print('DATA_DIR:', DATA_DIR)

In [ ]:
# Option B: Upload .conllu files directly
from google.colab import files
import os, shutil
os.makedirs('/content/repo/data', exist_ok=True)
uploaded = files.upload()  # select your .conllu files
for fn in uploaded:
    shutil.move(fn, f'/content/repo/data/{fn}')
DATA_DIR = 'data'
print('DATA_DIR:', DATA_DIR)
print('Files:   ', os.listdir(DATA_DIR))

## Cell 5 — Run windowed UID

`{DATA_DIR}` interpolates the Python variable into the shell command.

In [ ]:
!uv run python scripts/run_windowed_uid.py {DATA_DIR} gpt2 \
    --context document \
    --uid_unit token \
    --uid_level "(-10,+10)" \
    --generate_counterfactual \
    --fast --batch_size 128 \
    --output_dir outputs \
    --output_name window_uid_tok10.csv \
    --limit_docs 50 \
    --limit_sents_per_doc 12

## Cell 6 — Check coverage

In [ ]:
!uv run python scripts/check_window_coverage.py --csv outputs/window_uid_tok10.csv

## Cell 7 — Sweep windows (optional)

In [ ]:
!uv run python scripts/sweep_uid_windows.py {DATA_DIR} gpt2 \
    --windows "(-0,+0)" "(-5,+5)" "(-10,+10)" "(-20,+20)" \
    --context document \
    --generate_counterfactual \
    --fast --batch_size 128 \
    --output_dir outputs \
    --output_name sweep_results.csv \
    --limit_docs 50 --limit_sents_per_doc 12

## Cell 8 — Preview results

In [ ]:
import pandas as pd
df = pd.read_csv('outputs/window_uid_tok10.csv')
print(f'Rows: {len(df)}')
df.head()

## Cell 9 — Download results

## Cell 8b — Next-sentence analysis (discourse planning)

Measures whether the active/passive form of s_t changes the surprisal of s_{t+1}.
Uses word-level UID since that aligns with the linguistic research question.

In [ ]:
!uv run python scripts/run_next_sentence_uid.py {DATA_DIR} gpt2 \
    --context document \
    --uid_unit word \
    --uid_level sentence \
    --sent_offset 1 \
    --fast --batch_size 128 \
    --output_dir outputs \
    --output_name next_sent_uid.csv \
    --limit_docs 50 --limit_sents_per_doc 12

In [ ]:
from google.colab import files
files.download('outputs/window_uid_tok10.csv')
# Uncomment to also download sweep:
# files.download('outputs/sweep_results.csv')